# 1. Debug and Sanity

This notebook installs dependencies, finds the repo, checks the GPU, and runs the quick sanity tests plus a small parameter-count and activation-norm inspection.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/AtinChing/AttnResGPT-mini.git'
REPO_NAME = 'AttnResGPT-mini'

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

candidates = [
    Path(f'/content/{REPO_NAME}'),
    Path(f'/content/drive/MyDrive/{REPO_NAME}'),
    Path.cwd(),
]
repo_root = next((p for p in candidates if (p / 'requirements.txt').exists() and (p / 'src').exists()), None)

if repo_root is None:
    target = Path(f'/content/{REPO_NAME}')
    print(f'Cloning {REPO_URL} into {target} ...')
    subprocess.run(['git', 'clone', REPO_URL, str(target)], check=True)
    repo_root = target
else:
    print(f'Using existing repo at {repo_root}')

%cd {repo_root}
!pip -q install -r requirements.txt

In [ ]:
import torch

print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name:', torch.cuda.get_device_name(0))
    print('bf16_supported:', torch.cuda.is_bf16_supported())

In [ ]:
!pytest -q tests/test_shapes.py tests/test_masks.py tests/test_forward_pass.py tests/test_attnres.py

In [ ]:
from src.config import load_config
from src.data.dataset import build_dataloaders
from src.eval import build_model
from src.utils import count_parameters

for config_path in ['configs/baseline_t4_small.yaml', 'configs/attnres_t4_small.yaml']:
    cfg = load_config(config_path)
    tokenizer, _, _, _ = build_dataloaders(cfg)
    cfg.model.vocab_size = tokenizer.vocab_size
    model = build_model(cfg)
    print(config_path, count_parameters(model))

In [ ]:
import torch
from src.config import AttnResConfig, ModelConfig
from src.models.gpt_attnres import GPTAttnRes

cfg = ModelConfig(
    architecture='attnres',
    vocab_size=32,
    max_seq_len=32,
    d_model=64,
    n_layers=2,
    n_heads=4,
    d_ff=128,
    dropout=0.0,
    attnres=AttnResConfig(enabled=True, final_readout=True),
)
model = GPTAttnRes(cfg)
input_ids = torch.randint(0, cfg.vocab_size, (2, cfg.max_seq_len))
_, aux = model(input_ids, return_aux=True)
print('block_output_norms:', aux['block_output_norms'])
print('embedding_contribution:', aux['embedding_contribution'])
print('early_contribution:', aux['early_contribution'])
print('late_contribution:', aux['late_contribution'])
print('depth_attention_entropy:', aux['depth_attention_entropy'])